[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week5_first_cnn_STARTER.ipynb)


# Week 5 -- Your First CNN (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** first CNN on Oxford Pets — DummyClassifier baseline, F1 per class, confusion matrix

**Dataset:** Oxford-IIIT Pets (CNN baseline)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


In [ ]:
# -- Week 5 Imports --------------------------------------------------------
import os, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn          # neural network layers (Conv2d, Linear, etc.)
import torch.optim as optim    # optimisers (Adam, SGD)
from torch.utils.data import DataLoader   # batches Dataset samples
import torchvision.transforms as T        # image transforms
from sklearn.dummy import DummyClassifier # mandatory baseline
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from tqdm import tqdm          # progress bars for training loops

# device: where computation happens -- CPU or GPU
# ALL tensors and model parameters MUST be on the same device
# .to(device) moves them there -- call this for every tensor you create
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Tip: if device is cpu and you have a GPU, go to Runtime -> Change runtime type -> GPU')


In [ ]:
# -- Monday: Mandatory DummyClassifier Baseline ----------------------------
# RULE: Run this BEFORE training any real model.
# The DummyClassifier makes predictions without learning from features:
#   'most_frequent': always predicts the single most common class
#   'stratified':    predicts randomly with true class frequencies
# Your CNN MUST beat both. If it cannot, the pipeline has a bug.

from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report
import numpy as np

# Load flattened pixel arrays (sklearn needs 2D: n_samples x n_features)
print('Loading Oxford Pets arrays...')
try:
    X_train = np.load('outputs/X_train_flat.npy')  # (n_train, 150528)
    y_train = np.load('outputs/y_train.npy')        # (n_train,) labels 0-36
    X_test  = np.load('outputs/X_test_flat.npy')   # (n_test, 150528)
    y_test  = np.load('outputs/y_test.npy')         # (n_test,) labels 0-36
    print(f'  Train: {len(X_train)} | Test: {len(X_test)} | Classes: 37')
except FileNotFoundError:
    # Synthetic fallback -- replace with real dataset output from Week 3
    print('  Dataset not found -- using synthetic data')
    n_tr, n_te, n_cls = 500, 100, 37
    X_train = np.random.randn(n_tr, 50).astype('float32')
    X_test  = np.random.randn(n_te, 50).astype('float32')
    y_train = np.random.randint(0, n_cls, n_tr)
    y_test  = np.random.randint(0, n_cls, n_te)

# -- DummyClassifier: most_frequent ----------------------------------------
# .fit() only inspects the labels -- not the features
# .predict() always returns the most common class from training
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

# classification_report: per-class precision, recall, F1, support
# zero_division=0: avoids warnings when a class has zero predictions
print('\n=== DummyClassifier (most_frequent) ===')
print(classification_report(y_test, y_pred_dummy, zero_division=0))

# weighted F1: our primary metric -- weights each class by its sample count
# this accounts for class imbalance in the 37-class Pets dataset
wf1_floor = f1_score(y_test, y_pred_dummy, average='weighted', zero_division=0)
print(f'Weighted F1 FLOOR: {wf1_floor:.4f}')
print('Write this down. Your CNN must beat it -- or it has learned nothing.')


In [ ]:
# -- Tuesday: CNN Architecture -- Build and Understand It ------------------
# A CNN has three logical stages:
#   1. Feature extraction:   conv blocks learn what to look for in the image
#   2. Spatial aggregation:  collapse spatial dimensions to a feature vector
#   3. Classification:       map feature vector to class scores (logits)

import torch, torch.nn as nn

class PetsCNN(nn.Module):
    # Input:  (batch, 3, 224, 224) -- BCHW float32 normalised to [0,1]
    # Output: (batch, 37)          -- raw logits (NOT probabilities)
    # CrossEntropyLoss applies softmax internally -- do NOT add softmax here

    def __init__(self, num_classes=37):
        # Must call super().__init__() first -- sets up PyTorch internals
        super().__init__()

        # -- STAGE 1: Feature Extraction (3 convolutional blocks) -----------
        # Filter counts increase: 32 -> 64 -> 128
        # Early layers: simple features (edges, colours)
        # Later layers: complex features (textures, shapes, parts)
        self.features = nn.Sequential(

            # Block 1 -------------------------------------------------------
            # Conv2d(in_ch, out_ch, kernel, padding)
            #   in_channels=3:  input has 3 RGB channels
            #   out_channels=32: learns 32 different 3x3 filters
            #   kernel_size=3:  each filter sees a 3x3 pixel neighbourhood
            #   padding=1:      adds 1 zero-border so output H,W = input H,W
            nn.Conv2d(3, 32, kernel_size=3, padding=1),

            # BatchNorm2d normalises activations after the conv layer
            # Prevents 'internal covariate shift' that makes deep nets hard to train
            # Argument (32) must match the out_channels of the preceding Conv2d
            nn.BatchNorm2d(32),

            # ReLU activation: f(x) = max(0, x)
            # Introduces non-linearity -- without it, stacked linear layers
            # collapse to a single linear transformation (no added capacity)
            # inplace=True modifies the tensor directly (saves ~30% memory)
            nn.ReLU(inplace=True),

            # MaxPool2d(2,2): takes the max over each 2x2 region
            # Halves H and W: 224 -> 112
            # Makes features location-invariant over small pixel shifts
            nn.MaxPool2d(2, 2),  # 224 -> 112

            # Block 2 (in=32 must match Block 1 out=32) ---------------------
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 112 -> 56

            # Block 3 -------------------------------------------------------
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 56 -> 28
            # After Block 3: tensor shape is (batch, 128, 28, 28)
        )

        # -- STAGE 2+3: Aggregation and Classification ----------------------
        self.classifier = nn.Sequential(

            # AdaptiveAvgPool2d(1): collapses (batch,128,28,28) -> (batch,128,1,1)
            # 'Adaptive' means it works for ANY input size (not just 224x224)
            nn.AdaptiveAvgPool2d(1),

            # Flatten: removes the 1x1 spatial dims -> (batch, 128)
            # Each image is now a single 128-dimensional feature vector
            nn.Flatten(),

            # Dropout(0.5): randomly zeros 50% of activations DURING TRAINING
            # Forces redundancy -- the model cannot rely on any single feature
            # AUTOMATICALLY disabled during model.eval() -- no manual toggle needed
            nn.Dropout(0.5),

            # Linear(128, num_classes): maps features -> class logits
            # Output: (batch, 37) raw logits -- NOT probabilities
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # Data flow: (batch,3,224,224) -> features -> (batch,128,28,28)
        x = self.features(x)     # -> (batch, 128, 28, 28)
        x = self.classifier(x)   # -> (batch, 37)
        return x  # raw logits -- CrossEntropyLoss handles softmax

# Instantiate and move to device
model = PetsCNN(num_classes=37).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'PetsCNN | Parameters: {n_params:,} (ResNet-18 has 11.7M for comparison)')

# Shape trace -- verify the architecture BEFORE training
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 224, 224).to(device)  # fake batch
    output = model(dummy_input)
print(f'Shape trace: input {dummy_input.shape} -> output {output.shape}')
assert output.shape == (2, 37), f'SHAPE ERROR: expected (2,37), got {output.shape}'
print('Shape trace PASSED')


In [ ]:
# -- Wednesday: The Training Loop -- Four Steps, Every Batch ---------------
# The training loop implements supervised learning:
#   FOR each mini-batch from the DataLoader:
#     1. zero_grad()    -- clear accumulated gradients from last batch
#     2. forward pass   -- compute predictions and loss
#     3. loss.backward()-- compute gradients via backpropagation
#     4. optimizer.step()-- update weights to reduce the loss

import torch, torch.nn as nn, torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score

# CrossEntropyLoss: standard loss for multi-class classification
# Combines log-softmax + negative log-likelihood in one numerically stable step
# IMPORTANT: pass raw logits -- do NOT apply softmax to model output first
criterion = nn.CrossEntropyLoss()

# Adam optimiser with weight decay (L2 regularisation)
# lr=1e-3: learning rate -- too high -> unstable; too low -> slow convergence
# weight_decay=1e-4: penalises large weights to reduce overfitting
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()   # TRAINING MODE: enables Dropout, uses batch BatchNorm stats
    total_loss = 0.0
    for images, labels in tqdm(loader, desc='Training', leave=False):
        images = images.to(device)   # move images to GPU/CPU
        labels = labels.to(device)   # move labels to same device

        # Step 1: zero gradients -- MUST do this every batch
        # PyTorch accumulates gradients by default; skipping this contaminates
        # the current batch with gradients from all previous batches
        optimizer.zero_grad()

        # Step 2: forward pass -- compute predictions
        logits = model(images)            # shape: (batch, 37) raw logits

        # Compute loss: how wrong are the predictions?
        # criterion takes logits (batch,37) and labels (batch,) integer indices
        loss = criterion(logits, labels)  # returns a single scalar

        # Step 3: backward pass -- compute gradients via backpropagation
        # loss.backward() computes d(loss)/d(w) for every parameter w
        loss.backward()

        # Step 4: update weights -- optimizer uses gradients to adjust parameters
        # Adam: w_new = w_old - lr * (adaptive moment estimate of gradient)
        optimizer.step()

        # .item() converts 0-dim tensor to Python float (detaches from graph)
        total_loss += loss.item()

    return total_loss / len(loader)  # mean loss per batch

def evaluate(model, loader, device):
    model.eval()    # EVAL MODE: disables Dropout, uses running BatchNorm stats
    all_preds, all_labels = [], []
    with torch.no_grad():   # disable gradient computation (saves memory)
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)              # (batch, 37)
            preds  = logits.argmax(dim=1)       # class with highest logit
            all_preds.extend(preds.cpu().numpy())   # .cpu() before .numpy()
            all_labels.extend(labels.numpy())
    return all_preds, all_labels

# Training run
N_EPOCHS = 5
print(f'Training: {N_EPOCHS} epochs | Adam lr=1e-3 | CrossEntropyLoss')
print(f'DummyClassifier floor: {wf1_floor:.4f} -- must beat this')
print()
history = []
for epoch in range(1, N_EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    preds, labels = evaluate(model, val_loader, device)
    val_wf1 = f1_score(labels, preds, average='weighted', zero_division=0)
    history.append({'loss': tr_loss, 'wf1': val_wf1})
    print(f'  Epoch {epoch}/{N_EPOCHS}  loss={tr_loss:.4f}  val_wF1={val_wf1:.4f}')

best = max(h['wf1'] for h in history)
print(f'\nBest val weighted F1: {best:.4f}')
print(f'Beat the floor ({wf1_floor:.4f})? {"YES" if best > wf1_floor else "NO -- check pipeline"}')
torch.save(model.state_dict(), 'models/cnn_baseline.pth')
print('Saved: models/cnn_baseline.pth')


## Learning Objectives

By the end of this week, you will be able to:

- State your primary evaluation metric and deployment threshold before training any model
- Run the mandatory DummyClassifier baseline and explain what it reveals
- Build a CNN from scratch with three convolutional blocks and train it on Oxford-IIIT Pets
- Read the confusion matrix and classify the five most common error types
- Write an honest evaluation report that states whether the deployment threshold was met



---

## MONDAY -- "Problem Framing and the Mandatory Baseline"


Before training anything: commit to your evaluation metric in a markdown cell. For a 37-class imbalanced problem, accuracy is misleading. Weighted F1 accounts for class imbalance. Macro F1 treats all classes equally. Which is clinically appropriate? Then run the DummyClassifier. Its weighted F1 is the floor below which your model has learned nothing.


### Exercise 5.1 -- Metric commitment

Before training any model: write your primary metric, your secondary metric, and your deployment threshold. Justify each. You cannot change them after seeing results.


In [ ]:
# Exercise 5.1: Metric commitment
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


### Exercise 5.2 -- DummyClassifier baseline

Run DummyClassifier(strategy="most_frequent"). Run DummyClassifier(strategy="stratified"). Print classification_report for both. Which non-majority class has the highest recall in the stratified dummy? Why?


In [ ]:
# Exercise 5.2: DummyClassifier baseline
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Problem Framing and the Mandatory Baseline (scaffold) --
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report

# Mandatory baseline — always first
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train_flat, y_train)
y_pred_dummy = dummy.predict(X_test_flat)
print("DummyClassifier baseline:")
print(classification_report(y_test, y_pred_dummy, zero_division=0))


### Monday Morning Moment

*Slack — Monday, 2:30pm.*

**Ngozi Eze-Okafor:** Mandatory baseline done. DummyClassifier weighted F1: 0.034. Macro F1: 0.027.

**Yewande Adeyemi:** That's terrible.

**Dr. Kwame Asante:** That is the floor. The question is not whether your CNN beats it — it will. The question is by how much, and whether the gap is statistically significant.

**Ngozi Eze-Okafor:** I've committed to weighted F1 > 0.65 as the deployment threshold.

**Dr. Kwame Asante:** Write it in a markdown cell. Do not change it after you see results.

**Yewande Adeyemi:** Why 0.65? Why not 0.80?

**Ngozi Eze-Okafor:** Because this is a 37-class problem with class imbalance. 0.65 weighted F1 means the model meaningfully outperforms majority class prediction on most categories. 0.80 would be aspirational without evidence.

**Dr. Kwame Asante:** Good reasoning. Commit to 0.65. Revisit in Week 7 when you have transfer learning.




---

## TUESDAY -- "Building the CNN Architecture"


A CNN for image classification has three stages: feature extraction (convolutional blocks), spatial pooling (reduce dimensions), classification (fully connected). Each convolutional block: Conv2d → BatchNorm → ReLU → MaxPool. Three blocks, increasing filter counts (32→64→128), fixed kernel size 3×3.


### STOP AND TRACE

Before running: predict the output shape of each layer.

nn.Conv2d(3, 32, kernel_size=3, padding=1)

Input: (batch=4, C=3, H=224, W=224)
Output shape: ?

Work through: H_out = (H_in + 2×padding - kernel_size) / stride + 1
Why does padding=1 with kernel_size=3 preserve spatial dimensions?
Why this line exists: getting the shape wrong produces a silent bug — the output is a different size than expected, causing shape mismatches at the classifier.


In [ ]:
# Exercise 5.3: STOP AND TRACE — Conv2d output shape
# -------------------------------------------------------
# STOP AND TRACE: before running, write your expected output as a comment.
# Then run the cell and compare.

# YOUR PREDICTION:
# Expected output: ???

# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Building the CNN Architecture (scaffold) --
import torch.nn as nn

class PetsCNN(nn.Module):
    """Baseline CNN for Oxford-IIIT Pets classification.
    Architecture: 3 conv blocks → global average pool → classifier
    Input: (B, 3, 224, 224)  Output: (B, 37)
    """
    def __init__(self, num_classes=37):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 224 → 112
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 112 → 56
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 56 → 28
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))



---

## WEDNESDAY -- "Training Loop — What Actually Happens"


The training loop: forward pass → compute loss → backward pass → update weights. CrossEntropyLoss combines softmax + negative log-likelihood. Adam optimizer with lr=1e-3. One epoch = one pass through the full training set. Watch both training loss and validation loss — if validation loss starts rising while training loss falls, the model is overfitting.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Training Loop — What Actually Happens (scaffold) --
import torch
import torch.optim as optim

model = PetsCNN(num_classes=37).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = criterion(model(imgs), labels)
        outputs.backward()
        optimizer.step()
        total_loss += outputs.item()
    return total_loss / len(loader)



---

## THURSDAY -- "Error Analysis — Reading the Confusion Matrix"


A model that gets 63% accuracy is a model that gets 37% wrong. Read the errors. Which breed pairs are most confused? Abyssinian and Bengal look similar — the model may confuse them. That is genuine visual ambiguity. But if the model confuses Persian and Samoyed, something is wrong — they look nothing alike.


### Exercise 5.4 -- Error taxonomy

Read 20 misclassified images from your model. Classify each error: genuine visual ambiguity, label noise, class imbalance artefact, or preprocessing error. What fraction falls into each category?


In [ ]:
# Exercise 5.4: Error taxonomy
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Error Analysis — Reading the Confusion Matrix (scaffold) --
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(15,12))
sns.heatmap(cm, annot=False, xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.title("Confusion Matrix — Which breed pairs does the model confuse most?")
# Find the 5 most common off-diagonal errors
off_diag = cm.copy(); np.fill_diagonal(off_diag, 0)
for _ in range(5):
    i, j = np.unravel_index(off_diag.argmax(), off_diag.shape)
    print(f"  {class_names[i]} predicted as {class_names[j]}: {off_diag[i,j]} times")
    off_diag[i,j] = 0



---

## FRIDAY -- "The Friday Build: The Evaluation Report"


Write the honest evaluation report. Report: model architecture, primary metric, DummyClassifier floor, your model's result, whether the deployment threshold was met, the five most common error types, and three recommended next steps. If the threshold was not met, say so clearly. An honest report that says "not ready" is more valuable than an optimistic one that hides failures.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: The Evaluation Report (scaffold) --
# The evaluation report must contain:
# 1. Metric commitment (written before training)
# 2. DummyClassifier baseline (the floor)
# 3. Your model result (weighted F1, per-class breakdown)
# 4. Whether the deployment threshold was met: YES or NO
# 5. The 5 most common error types with clinical translation
# 6. Three next steps


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Dr. Kwame Asante:** Evaluation report. Did the threshold meet?

**Ngozi Eze-Okafor:** Weighted F1: 0.54. Threshold: 0.65. Not met. Three epochs, batch size 32, Adam lr=1e-3. The model meaningfully outperforms the dummy baseline on 29 of 37 classes, but underperforms on the 8 underrepresented classes from Week 2.

**Dr. Kwame Asante:** What does that mean for deployment?

**Ngozi Eze-Okafor:** Not ready. For the rare breeds — and by analogy, for rare tumour subtypes in Week 7 — the model is worse than an informed guess.

**Yewande Adeyemi:** Transfer learning will fix the rare class problem. Week 7.

**Dr. Kwame Asante:** It will help. It will not fix it. Write the distinction in the report.



## YOUR WEEK 5 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I committed to my evaluation metric before training — and did not change it after seeing results.
- [ ] I ran the DummyClassifier baseline before any real model and can state its weighted F1 from memory.
- [ ] I can predict the output shape of a Conv2d layer given input dimensions without running the code.
- [ ] I classified my model's errors by type — not just by count.
- [ ] My evaluation report says honestly whether the deployment threshold was met.


## Challenge Task

> **Core path:** Implement learning rate scheduling (CosineAnnealingLR). Train for 10 epochs with and without scheduling. Plot both training and validation loss curves. Does scheduling help? By how much?

> **Advanced path:** Implement early stopping based on validation loss. What is the optimal epoch for your CNN? Compare to the fixed 5-epoch run.


## Common Mistakes

**Choosing the metric after seeing results:** Commit to weighted F1, macro F1, or AUC before training. Do not switch to the metric that makes your model look best.

**Skipping the DummyClassifier baseline:** A model that gets 55% accuracy on a 37-class problem may still be performing worse than majority class prediction. Always establish the floor.

**Reporting accuracy on imbalanced data:** With 8 underrepresented classes, accuracy is dominated by the majority classes. Weighted F1 is the minimum honest metric. Per-class recall is the clinically honest metric.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
